<a href="https://colab.research.google.com/github/ODBapp/2025_NODASS_workshop/blob/main/src/odb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/ODBapp/2025_NODASS_workshop.git

# 地理資訊

In [10]:
%cd 2025_NODASS_workshop/src/geo

[Errno 2] No such file or directory: '2025_NODASS_workshop/src/geo'
/content/2025_NODASS_workshop/src/geo


In [11]:
import csv, json
import xml.etree.ElementTree as ET

In [12]:
# === 轉換時間為ISO8601 ===
def toISO8601(datetime_string):
    return datetime_string.strip().replace('/', '-').replace(' ', 'T')

In [13]:
# === GeoJSON輸出 ===
def export_geojson(out_path, coords, pt_properties=None, line=True, line_properties=None):
    pt_properties = pt_properties or []
    line_properties = line_properties or {}

    features = []
    for idx, coord in enumerate(coords):
        props = pt_properties[idx] if idx < len(pt_properties) else {}
        features.append({
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": coord},
            "properties": props
        })

    if line:
        features.append({
            "type": "Feature",
            "geometry": {
                "type": "LineString",
                "coordinates": coords
            },
            "properties": line_properties
        })

    geojson = {
        "type": "FeatureCollection",
        "features": features
    }
    with open(out_path, 'w', encoding='utf-8') as f:
        # json.dump(geojson, f) # 無排版，檔案較小(15KB)
        json.dump(geojson, f, indent=2) # 排版，檔案較大 (30KB)

In [14]:
# === KML輸出 ===
def export_kml(out_path, coords, pt_descriptions=None, pt_names=None, line=True, line_properties=None, line_name='線段'):
    pt_descriptions = pt_descriptions or []
    pt_names = pt_names or []
    line_properties = line_properties or {}

    kml = ET.Element("kml", xmlns="http://www.opengis.net/kml/2.2")
    document = ET.SubElement(kml, "Document")

    names = pt_names
    if len(coords) > len(names):
        names = [str(i + 1) for i in range(len(coords))]

    for idx, coord in enumerate(coords):
        placemark = ET.SubElement(document, "Placemark")
        ET.SubElement(placemark, "name").text = names[idx]
        if idx < len(pt_descriptions):
            ET.SubElement(placemark, "description").text = pt_descriptions[idx]

        point = ET.SubElement(placemark, "Point")
        ET.SubElement(point, "coordinates").text = f"{coord[0]},{coord[1]}"

    if line:
        placemark = ET.SubElement(document, "Placemark")
        ET.SubElement(placemark, "name").text = line_name
        ET.SubElement(placemark, "description").text = line_properties

        linestring = ET.SubElement(placemark, "LineString")
        ET.SubElement(linestring, "tessellate").text = "1"
        coords_text = ' '.join(f"{coord[0]},{coord[1]}" for coord in coords)
        ET.SubElement(linestring, "coordinates").text = coords_text

    ET.indent(kml, space="  ") # 排版方便閱讀，但檔案大小增加 (17KB vs 22KB)
    ET.ElementTree(kml).write(out_path, encoding='utf-8-sig', xml_declaration=True)

In [15]:
# === 路徑檔名設定 ===
csv_path = '202507021522_export_GDP_22943_result.csv'
geojson_path = 'output.json'
kml_path = 'output.kml'

# === 讀取CSV ===
coords = []
times = []
json_pt_properties = []
kml_descriptions = []
total_sst = 0.0
count = 0

with open(csv_path, newline='', encoding='utf-8') as csvfile:
    reader = csv.DictReader(csvfile)
    for row in reader:
        try:
            lon = float(row['CenterLongitude'])
            lat = float(row['CenterLatitude'])
            time = toISO8601(row['time'])
            sst = float(row['sst'])

            coords.append([lon, lat])
            times.append(time)
            json_pt_properties.append({"time": time, "sst": sst})
            kml_descriptions.append(f"SST: {sst}<br/>lon: {lon}<br/>lat: {lat}")

            total_sst += sst
            count += 1
        except (ValueError, KeyError):
            continue

In [16]:
# 加入線段資料
mean_sst = round(10* total_sst / count) / 10

json_ln_properties = {
    "start": times[0],
    "end": times[-1],
    "sst_avg": mean_sst
}
kml_ln_properties = (
    f"Start: {times[0]}<br/>"
    f"End: {times[-1]}<br/>"
    f"Mean SST: {mean_sst}"
)

In [17]:
# 輸出
export_geojson(geojson_path, coords, json_pt_properties, True, json_ln_properties)
export_kml(kml_path, coords, kml_descriptions, [], True, kml_ln_properties, 'GDP軌跡')

# 時間序列

In [ ]:
%cd 2025_NODASS_workshop/src/timeseries

# 海洋熱浪

In [ ]:
%cd 2025_NODASS_workshop/src/mhw

# 生物資料

In [ ]:
%cd 2025_NODASS_workshop/src/bio